# RYDEXA: Empirical Bike-Sharing Demand Forecasting ML Pipeline
### Reproducible Google Colab Training, Evaluation & Artifact Export Pipeline
**Dataset**: UCI Machine Learning Repository — Capital Bikeshare (17,379 hourly observations)
**Objective**: Forecast hourly fleet rental demand (`cnt`) without target leakage, using a chronological evaluation split.

--- 
### Pipeline Architecture
1. **Data Ingestion**: Clean & validate 17,379 records (`casual + registered == cnt`).
2. **Target Leakage Prevention**: Strictly drop `casual` and `registered` from inputs.
3. **Chronological Splitting**: Train on 2011 to Q3 2012 (15,211 hrs), evaluate on unseen Q4 2012 (2,168 hrs).
4. **Model Comparison**: Baseline vs Ridge vs Random Forest vs HistGradientBoosting.
5. **Artifact Export**: Export `model.joblib`, `metrics.json`, `feature_schema.json`, and `analytics_summary.json`, then bundle them as `rydexa_artifacts.zip`.
6. **Backend Import**: Validate and install the zip with `python backend/import_artifacts.py <zip>`. Colab runs separately and is **not** connected to the local FastAPI server, so the import is a deliberate manual step.

In [ ]:
# [Cell 1] Environment Setup & Dependencies
!pip install -q scikit-learn pandas numpy joblib requests
import os, zipfile, json, urllib.request, shutil
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
import joblib
print('Dependencies loaded successfully.')

In [ ]:
# [Cell 2] Download Official UCI Bike Sharing Dataset
DATASET_URL = 'https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip'
os.makedirs('dataset', exist_ok=True)
zip_path = 'dataset/bike_sharing.zip'
if not os.path.exists('dataset/hour.csv'):
    print('Downloading UCI dataset archive...')
    req = urllib.request.Request(DATASET_URL, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as resp, open(zip_path, 'wb') as f:
        f.write(resp.read())
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('dataset')
df = pd.read_csv('dataset/hour.csv')
print(f'Loaded {len(df)} records. Columns: {list(df.columns)}')
# Integrity Verification
assert (df['casual'] + df['registered'] == df['cnt']).all(), 'Target integrity failure!'
print('Integrity confirmed: casual + registered == cnt across all 17,379 observations.')

In [ ]:
# [Cell 3] Chronological Train / Test Split
df['dteday'] = pd.to_datetime(df['dteday'])
df = df.sort_values(by=['dteday', 'hr']).reset_index(drop=True)
# Chronological Split: 2011-01 to 2012-09 (train) vs 2012-10 to 2012-12 (held-out test)
split_date = pd.to_datetime('2012-10-01')
df_train = df[df['dteday'] < split_date].copy()
df_test = df[df['dteday'] >= split_date].copy()
print(f'Train observations: {len(df_train)} ({df_train["dteday"].min().strftime("%Y-%m-%d")} to {df_train["dteday"].max().strftime("%Y-%m-%d")})')
print(f'Held-out Test observations: {len(df_test)} ({df_test["dteday"].min().strftime("%Y-%m-%d")} to {df_test["dteday"].max().strftime("%Y-%m-%d")})')

In [ ]:
# [Cell 4] Leakage-Free Feature Engineering
def engineer_features(data):
    X = pd.DataFrame(index=data.index)
    for col in ['season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed']:
        X[col] = data[col]
    X['hr_sin'] = np.sin(2 * np.pi * data['hr'] / 24.0)
    X['hr_cos'] = np.cos(2 * np.pi * data['hr'] / 24.0)
    X['mnth_sin'] = np.sin(2 * np.pi * data['mnth'] / 12.0)
    X['mnth_cos'] = np.cos(2 * np.pi * data['mnth'] / 12.0)
    X['is_commute_peak'] = data['hr'].isin([7, 8, 9, 17, 18, 19]).astype(int) * data['workingday']
    X['is_adverse_weather'] = (data['weathersit'] >= 3).astype(int)
    X['temp_discomfort'] = (data['atemp'] - data['temp']).abs()
    return X

X_train = engineer_features(df_train)
y_train = df_train['cnt']
X_test = engineer_features(df_test)
y_test = df_test['cnt']
print(f'Engineered {X_train.shape[1]} features: {list(X_train.columns)}')

In [ ]:
# [Cell 5] Baseline vs Candidate Regressors Evaluation
lookup = df_train.groupby(['hr', 'workingday'])['cnt'].median()
baseline_preds = np.array([lookup.get((r['hr'], r['workingday']), df_train['cnt'].median()) for _, r in df_test.iterrows()])

print('Training candidate models...')
ridge = Ridge(alpha=10.0).fit(X_train, y_train)
rf = RandomForestRegressor(n_estimators=100, max_depth=16, random_state=42, n_jobs=-1).fit(X_train, y_train)
hgbr = HistGradientBoostingRegressor(max_iter=250, learning_rate=0.08, max_leaf_nodes=31, random_state=42).fit(X_train, y_train)

models = {
    'Historical Median Baseline': baseline_preds,
    'Ridge Regression': np.clip(ridge.predict(X_test), 0, None),
    'Random Forest Regressor': np.clip(rf.predict(X_test), 0, None),
    'Gradient Boosting Regressor': np.clip(hgbr.predict(X_test), 0, None)
}

comparison = []
for name, preds in models.items():
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    comparison.append({'model_name': name, 'mae': round(mae, 2), 'rmse': round(rmse, 2), 'r2': round(r2, 4), 'is_champion': name == 'Gradient Boosting Regressor'})

eval_df = pd.DataFrame(comparison)
display(eval_df)

In [ ]:
# [Cell 6] Precompute Historical Dataset Aggregations
analytics = {
    'dataset_summary': {
        'total_records': int(len(df)),
        'time_range': f"{df['dteday'].min().strftime('%Y-%m-%d')} to {df['dteday'].max().strftime('%Y-%m-%d')}",
        'total_rentals': int(df['cnt'].sum()),
        'mean_hourly_rentals': round(float(df['cnt'].mean()), 2),
        'median_hourly_rentals': int(df['cnt'].median()),
        'max_hourly_rentals': int(df['cnt'].max()),
        'std_hourly_rentals': round(float(df['cnt'].std()), 2)
    },
    'hourly_pattern': [
        {
            'hour': h,
            'workingday_avg': round(float(df[(df['workingday'] == 1) & (df['hr'] == h)]['cnt'].mean()), 1),
            'non_workingday_avg': round(float(df[(df['workingday'] == 0) & (df['hr'] == h)]['cnt'].mean()), 1),
            'overall_avg': round(float(df[df['hr'] == h]['cnt'].mean()), 1)
        }
        for h in range(24)
    ],
    'season_pattern': [
        {
            'season_id': s,
            'season_name': {1: 'Spring', 2: 'Summer', 3: 'Fall', 4: 'Winter'}.get(s),
            'mean_hourly': round(float(df[df['season'] == s]['cnt'].mean()), 1),
            'median_hourly': round(float(df[df['season'] == s]['cnt'].median()), 1),
            'total_rentals': int(df[df['season'] == s]['cnt'].sum())
        }
        for s in [1, 2, 3, 4]
    ],
    'weather_pattern': [
        {
            'weathersit_id': w,
            'weather_name': {1: 'Clear / Partly Cloudy', 2: 'Mist / Cloudy', 3: 'Light Snow / Light Rain', 4: 'Heavy Rain / Ice Pellets'}.get(w),
            'mean_hourly': round(float(df[df['weathersit'] == w]['cnt'].mean()), 1),
            'sample_count': int(len(df[df['weathersit'] == w]))
        }
        for w in [1, 2, 3, 4]
    ],
    'temperature_pattern': [
        {'temp_range': '< 10°C (Cold)', 'mean_hourly': round(float(df[df['temp'] * 41 < 10]['cnt'].mean()), 1), 'sample_count': int(len(df[df['temp'] * 41 < 10]))},
        {'temp_range': '10°C - 20°C (Mild)', 'mean_hourly': round(float(df[(df['temp'] * 41 >= 10) & (df['temp'] * 41 < 20)]['cnt'].mean()), 1), 'sample_count': int(len(df[(df['temp'] * 41 >= 10) & (df['temp'] * 41 < 20)]))},
        {'temp_range': '20°C - 30°C (Warm/Optimal)', 'mean_hourly': round(float(df[(df['temp'] * 41 >= 20) & (df['temp'] * 41 < 30)]['cnt'].mean()), 1), 'sample_count': int(len(df[(df['temp'] * 41 >= 20) & (df['temp'] * 41 < 30)]))},
        {'temp_range': '30°C+ (Hot)', 'mean_hourly': round(float(df[df['temp'] * 41 >= 30]['cnt'].mean()), 1), 'sample_count': int(len(df[df['temp'] * 41 >= 30]))}
    ]
}
print('Analytics summary precomputed.')

In [ ]:
# [Cell 7] Export Complete Backend Artifact Suite
os.makedirs('artifacts', exist_ok=True)

# 1. Save Trained Model
joblib.dump(hgbr, 'artifacts/model.joblib')

# 2. Save Analytics Summary
with open('artifacts/analytics_summary.json', 'w') as f:
    json.dump(analytics, f, indent=2)

# 3. Save Evaluation Metrics
best_m = [m for m in comparison if m['is_champion']][0]
metrics_dict = {
    'dataset_name': 'UCI Machine Learning Repository - Bike Sharing Dataset (hour.csv)',
    'source_url': 'https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset',
    'historical_context': 'Capital Bikeshare system, Washington D.C., years 2011-2012',
    'split_strategy': 'Chronological (Train: 2011-01-01 to 2012-09-30; Held-out Test: 2012-10-01 to 2012-12-31)',
    'training_samples': len(df_train),
    'test_samples': len(df_test),
    'target': 'cnt (total hourly rentals)',
    'champion_model': 'Gradient Boosting Regressor',
    'champion_metrics': {'mae': best_m['mae'], 'rmse': best_m['rmse'], 'r2': best_m['r2']},
    'models_evaluated': comparison,
    'disclosures': [
        'Dataset represents Capital Bikeshare in Washington D.C. (2011-2012), not modern dockless e-bike operations.',
        'Predictions reflect historical behavioral patterns under given meteorological and calendar conditions.',
        'Excludes registered and casual sub-counts to eliminate target leakage.'
    ]
}
with open('artifacts/metrics.json', 'w') as f:
    json.dump(metrics_dict, f, indent=2)

# 4. Save Feature Constraints Schema
feature_schema = {
    'features': list(X_train.columns),
    'input_ranges': {
        'season': {'type': 'integer', 'min': 1, 'max': 4, 'labels': {1: 'Spring', 2: 'Summer', 3: 'Fall', 4: 'Winter'}},
        'hr': {'type': 'integer', 'min': 0, 'max': 23, 'description': 'Hour of the day (0 to 23)'},
        'holiday': {'type': 'integer', 'min': 0, 'max': 1, 'description': '0 = No, 1 = Holiday'},
        'weekday': {'type': 'integer', 'min': 0, 'max': 6, 'labels': {0: 'Sunday', 1: 'Monday', 2: 'Tuesday', 3: 'Wednesday', 4: 'Thursday', 5: 'Friday', 6: 'Saturday'}},
        'workingday': {'type': 'integer', 'min': 0, 'max': 1, 'description': '1 if neither weekend nor holiday, else 0'},
        'weathersit': {'type': 'integer', 'min': 1, 'max': 4, 'labels': {1: 'Clear/Partly Cloudy', 2: 'Mist/Cloudy', 3: 'Light Snow/Rain', 4: 'Heavy Rain/Ice'}},
        'temp_c': {'type': 'float', 'min': -10.0, 'max': 45.0, 'unit': 'Celsius', 'norm_formula': 'temp = (celsius + 8) / 47 or celsius / 41'},
        'humidity': {'type': 'float', 'min': 0.0, 'max': 100.0, 'unit': '%', 'norm_formula': 'hum = humidity / 100'},
        'windspeed_kmh': {'type': 'float', 'min': 0.0, 'max': 70.0, 'unit': 'km/h', 'norm_formula': 'windspeed = windspeed_kmh / 67'}
    },
    'target': 'cnt (total hourly rentals)',
    'excluded_features': ['casual', 'registered', 'instant', 'dteday']
}
with open('artifacts/feature_schema.json', 'w') as f:
    json.dump(feature_schema, f, indent=2)

# 5. Create Zip Bundle for 1-Click Deployment
shutil.make_archive('rydexa_artifacts', 'zip', 'artifacts')
print('All 4 artifacts generated and packaged into rydexa_artifacts.zip!')

# Optional Google Colab download trigger
try:
    from google.colab import files
    files.download('rydexa_artifacts.zip')
    print('Colab download initiated.')
except ImportError:
    print('Local environment detected: rydexa_artifacts.zip is ready in current directory.')
# 6. Backend import is a separate, manual step on the machine running FastAPI
print('\n--- Next steps (outside Colab) ---')
print('1. Download rydexa_artifacts.zip from the Colab file browser (or the automatic download above).')
print('2. From the repository root run:')
print('   python backend/import_artifacts.py ~/Downloads/rydexa_artifacts.zip')
print('3. Restart the backend: python backend/run.py')
print('The importer validates every file, backs up backend/artifacts/, and installs the new model.')
print('Colab is NOT connected to the running backend; this ZIP is the hand-off.')
